<a href="https://colab.research.google.com/github/estebanvillalba2007-source/curso_programacion_cientifica/blob/main/Taller_evaluativo_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller evaluativo 03 — NumPy

# Ejercicio 1 — Crear los arreglos y leer su etiqueta
Convierta los datos crudos en un arreglo y descríbalo, como quien mira cuántas filas y columnas tiene una hoja de cálculo antes de trabajar con ella.

In [1]:
import numpy as np

lecturas_crudas = np.array([
    [10.0, 14.5, 22.0, 30.5, 28.0, 18.0, 9.5],
    [12.0, 20.0, 35.0, 48.0, 41.0, 25.0, 14.0],
    [8.0, 11.0, 19.0, 27.0, 33.0, 21.0, 6.0],
    [15.0, 24.0, 31.0, 44.0, 52.0, 36.0, 17.0],
    [5.0, 9.0, 16.0, 22.0, 45.0, 30.0, 4.0]
])

forma = lecturas_crudas.shape
dimensiones = lecturas_crudas.ndim
total = lecturas_crudas.size
tipo = lecturas_crudas.dtype

dias = np.arange(1, 8)

pendientes = np.array([1.2, 0.9, 1.1, 1.0, 1.3])
interceptos = np.array([-0.5, 1.0, -2.0, 0.5, -1.5])

print("Forma:", forma)
print("Dimensiones:", dimensiones)
print("Total:", total)
print("Tipo:", tipo)
print("Días:", dias)
print("Pendientes:", pendientes)
print("Interceptos:", interceptos)

Forma: (5, 7)
Dimensiones: 2
Total: 35
Tipo: float64
Días: [1 2 3 4 5 6 7]
Pendientes: [1.2 0.9 1.1 1.  1.3]
Interceptos: [-0.5  1.  -2.   0.5 -1.5]


# Ejercicio 2 — Señalar celdas y recortar bloques
Use indexación y slicing para leer partes de la tabla. Recuerde que Python cuenta desde 0: el sensor SBC-A es la fila 0 y el día 1 es la columna 0.

In [2]:
lectura_c_dia4 = lecturas_crudas[2, 3]

ultimo_primer = lecturas_crudas[-1, 0]

dia5 = lecturas_crudas[:, 4]

semana_b = lecturas_crudas[1, :]

bloque = lecturas_crudas[1:4, 2:6]

print("Lectura de SBC-C en el día 4:", lectura_c_dia4)
print("Último sensor en el día 1:", ultimo_primer)
print("Lecturas del día 5:", dia5)
print("Semana de SBC-B:", semana_b)
print("Bloque de SBC-B a SBC-D, días 3 a 6:\n", bloque)
print("Forma del bloque:", bloque.shape)

Lectura de SBC-C en el día 4: 27.0
Último sensor en el día 1: 5.0
Lecturas del día 5: [28. 41. 33. 52. 45.]
Semana de SBC-B: [12. 20. 35. 48. 41. 25. 14.]
Bloque de SBC-B a SBC-D, días 3 a 6:
 [[35. 48. 41. 25.]
 [19. 27. 33. 21.]
 [31. 44. 52. 36.]]
Forma del bloque: (3, 4)


# Ejercicio 3 — Limpiar una campaña con máscaras booleanas
Esta es la serie de ocho lecturas de campaña de SBC-A del taller anterior, todas en µg/m³ y sin calibrar:
campana_sbc_a = [10.0, -999.0, 25.0, 40.0, -999.0, 8.0, 52.0, 600.0]
Una lectura es útil cuando está entre 0 y 500.0 inclusive: eso descarta tanto el centinela -999.0 como el ruido de más de 500.0.

In [3]:
campana_sbc_a = np.array([
    10.0, -999.0, 25.0, 40.0,
    -999.0, 8.0, 52.0, 600.0
])

faltantes = campana_sbc_a == -999.0

es_util = (
    (campana_sbc_a >= 0) &
    (campana_sbc_a <= 500.0)
)

lecturas_utiles = campana_sbc_a[es_util]

promedio_utiles = lecturas_utiles.mean()

print("Cantidad de lecturas faltantes:", faltantes.sum())
print("Lecturas útiles:", lecturas_utiles)
print("Promedio de lecturas útiles:", promedio_utiles)

Cantidad de lecturas faltantes: 2
Lecturas útiles: [10. 25. 40.  8. 52.]
Promedio de lecturas útiles: 27.0


# Ejercicio 4 — Calibrar todo con operaciones vectorizadas y broadcasting
Calibre las 35 lecturas de la red de un solo golpe, sin recorrerlas. Cada sensor tiene su propia pendiente y su propio intercepto, así que hay que alinear los arreglos de distinta forma.

In [4]:
pendientes_col = pendientes.reshape(5, 1)

interceptos_col = interceptos.reshape(5, 1)

calibradas = (
    lecturas_crudas * pendientes_col
    + interceptos_col
)

campana_calibrada = lecturas_utiles * 1.2 - 0.5

promedio_calibrado_sbc_a = campana_calibrada.mean()

print("Forma de calibradas:", calibradas.shape)
print("Primera fila calibrada:", calibradas[0])
print("Campaña calibrada de SBC-A:", campana_calibrada)
print(
    "Promedio de campaña calibrada de SBC-A:",
    promedio_calibrado_sbc_a
)

Forma de calibradas: (5, 7)
Primera fila calibrada: [11.5 16.9 25.9 36.1 33.1 21.1 10.9]
Campaña calibrada de SBC-A: [11.5 29.5 47.5  9.1 61.9]
Promedio de campaña calibrada de SBC-A: 31.9


# Ejercicio 5 — Resumir la semana con estadística por eje
Con calibradas de aquí en adelante, resuma la red completa y por dirección. Recuerde: axis=0 resume por día (un valor por columna) y axis=1 resume por sensor (un valor por fila).

In [5]:
promedio_general = calibradas.mean()

promedio_por_sensor = calibradas.mean(axis=1)

promedio_por_dia = calibradas.mean(axis=0)

maximo_por_sensor = calibradas.max(axis=1)

media_sobre_limite = calibradas[calibradas > 35.4].mean()

print("Promedio general:", np.round(promedio_general, 2))

print(
    "Promedio por sensor:",
    np.round(promedio_por_sensor, 2)
)

print(
    "Promedio por día:",
    np.round(promedio_por_dia, 2)
)

print(
    "Máximo por sensor:",
    np.round(maximo_por_sensor, 2)
)

print(
    "Promedio de lecturas mayores que 35.4:",
    np.round(media_sobre_limite, 2)
)

Promedio general: 24.11
Promedio por sensor: [22.21 26.07 17.64 31.79 22.83]
Promedio por día: [10.12 16.14 25.62 35.92 42.96 27.94 10.06]
Máximo por sensor: [36.1 44.2 34.3 52.5 57. ]
Promedio de lecturas mayores que 35.4: 43.28


# Ejercicio 6 — Funciones universales y reemplazo con máscara
Mida cuánto se pasó cada sensor del límite de 35.4 µg/m³ (la frontera entre Moderada y Dañina para grupos sensibles), celda por celda.

In [7]:
excedente = np.maximum(calibradas - 35.4, 0)

excedente_por_sensor = np.round(
    excedente.sum(axis=1), 1
)

desvio_sensor_b = np.round(
    np.abs(calibradas[1] - 35.4), 1
)

en_alerta = calibradas.copy()

en_alerta[en_alerta <= 35.4] = 0.0

print("Excedente por sensor:", excedente_por_sensor)

print("Distancia absoluta de SBC-B al límite:", desvio_sensor_b)

print(
    "Celdas de en_alerta mayores que 0:",
    (en_alerta > 0).sum()
)


Excedente por sensor: [ 0.7 11.3  0.  27.3 23.7]
Distancia absoluta de SBC-B al límite: [23.6 16.4  2.9  8.8  2.5 11.9 21.8]
Celdas de en_alerta mayores que 0: 8


# Ejercicio 7 — Integración: el informe de la red
El último ejercicio junta todo: máscaras sobre los datos calibrados para contar cuántas lecturas cayeron en cada categoría de la escala oficial, y la identificación del día más contaminado.

In [8]:
buenas = (calibradas <= 12.0).sum()

moderadas = (
    (calibradas > 12.0) &
    (calibradas <= 35.4)
).sum()

daninas_sensibles = (
    (calibradas > 35.4) &
    (calibradas <= 55.4)
).sum()

daninas_salud = (calibradas > 55.4).sum()

dia_pico = dias[
    promedio_por_dia == promedio_por_dia.max()
][0]

promedio_dia_pico = promedio_por_dia.max()

print(
    f"Lecturas por categoría: Buena={buenas}, "
    f"Moderada={moderadas}, "
    f"Dañina para grupos sensibles={daninas_sensibles}, "
    f"Dañina para la salud={daninas_salud}"
)

print(
    f"Día más contaminado: {dia_pico} con "
    f"{np.round(promedio_dia_pico, 1)} ug/m3"
)

Lecturas por categoría: Buena=9, Moderada=18, Dañina para grupos sensibles=7, Dañina para la salud=1
Día más contaminado: 5 con 43.0 ug/m3
